# Predictor agreement

In [7]:
import numpy as np
from scipy import stats
from scipy.stats import spearmanr
import pingouin as pg
import pandas as pd

In [11]:
# read in data
families = ['gpt_5', 'claude_4_5', 'gemini_3', 'qwen_3', 'gemma_3']


name_dict = {
    "openai/gpt-5-mini_medium": "GPT-5 Mini",
    "openai/gpt-5-nano_medium": "GPT-5 Nano",
    "openai/gpt-5.2_medium": "GPT-5.2",
    "anthropic/claude-haiku-4.5_medium": "Claude Haiku 4.5",
    "anthropic/claude-opus-4.5_medium": "Claude Opus 4.5",
    "anthropic/claude-sonnet-4.5_medium": "Claude Sonnet 4.5",
    "google/gemini-3-flash-preview_medium": "Gemini 3 Flash",
    "google/gemini-3-pro-preview_medium": "Gemini 3 Pro",
    "Qwen/Qwen3-0.6B_True": "Qwen 3 0.6B",
    "Qwen/Qwen3-1.7B_True": "Qwen 3 1.7B",
    "Qwen/Qwen3-14B_True": "Qwen 3 14B",
    "Qwen/Qwen3-32B_True": "Qwen 3 32B",
    "Qwen/Qwen3-4B_True": "Qwen 3 4B",
    "Qwen/Qwen3-8B_True": "Qwen 3 8B",
    "google/gemma-3-12b-it": "Gemma 3 12B",
    "google/gemma-3-1b-it": "Gemma 3 1B",
    "google/gemma-3-27b-it": "Gemma 3 27B",
    "google/gemma-3-4b-it": "Gemma 3 4B",
}

# Family display names
family_display_names = {
    'qwen_3': 'Qwen 3',
    'gemma_3': 'Gemma 3',
    'gpt_5': 'GPT-5',
    'claude_4_5': 'Claude 4.5',
    'gemini_3': 'Gemini 3',
}

all_breakdown_data = []
for f in families:
    df_ = pd.read_csv(f"../experiments/dewi_upload/{f}_predictions_simulatability_breakdown_by_predictor.csv")
    df_['family'] = f
    all_breakdown_data.append(df_)

df_breakdown = pd.concat(all_breakdown_data, ignore_index=True)
df_breakdown['short_name'] = df_breakdown['model'].map(name_dict)




In [20]:
df_breakdown

,predictor,predictor_index,model,total,with_explanation_accuracy,without_explanation_accuracy,simulatability_gain,gain_ci_lower,gain_ci_upper,with_ci_lower,...,recall_without_ci_lower,recall_without_ci_upper,diff_pct,both_correct,both_wrong,only_with_correct,only_without_correct,family,short_name,ref_model_idx
0,openai/gpt-oss-20b,0,openai/gpt-5-mini_medium,6843,80.403332,68.566418,11.836914,10.651473,13.088272,79.309863,...,62.047829,65.243269,23.761508,4286,935,1216,406,gpt_5,GPT-5 Mini,0
1,openai/gpt-oss-20b,0,openai/gpt-5-nano_medium,6093,78.385032,65.846053,12.538979,11.137123,13.988125,77.225408,...,58.559347,62.136686,26.210405,3619,924,1157,393,gpt_5,GPT-5 Nano,1
2,openai/gpt-oss-20b,0,openai/gpt-5.2_medium,6837,81.102823,68.816732,12.286090,11.062127,13.467625,79.947077,...,62.890526,66.045000,23.314319,4330,917,1215,375,gpt_5,GPT-5.2,2
3,google/gemma-3-27b-it,1,openai/gpt-5-mini_medium,6843,80.257197,67.821131,12.436066,11.237248,13.784969,79.195213,...,66.563556,69.695321,23.761508,4311,1021,1181,330,gpt_5,GPT-5 Mini,0
4,google/gemma-3-27b-it,1,openai/gpt-5-nano_medium,6093,77.843427,64.418185,13.425242,12.084174,14.804442,76.585134,...,61.781488,65.470075,26.210405,3637,1062,1106,288,gpt_5,GPT-5 Nano,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,google/gemini-3-flash-preview,3,google/gemma-3-4b-it,6514,82.821615,77.586736,5.234879,4.240458,6.246305,81.842177,...,79.671964,81.887016,13.355849,4709,774,686,345,gemma_3,Gemma 3 4B,17
86,openai/gpt-5-mini,4,google/gemma-3-12b-it,6231,80.404429,76.263842,4.140587,3.141696,5.275668,79.416388,...,76.853053,79.517019,15.519178,4419,888,591,333,gemma_3,Gemma 3 12B,14
87,openai/gpt-5-mini,4,google/gemma-3-1b-it,6859,69.908150,67.677504,2.230646,1.141415,3.269600,68.651256,...,72.954377,75.702171,15.381251,4124,1546,671,518,gemma_3,Gemma 3 1B,15
88,openai/gpt-5-mini,4,google/gemma-3-27b-it,6997,79.448335,75.132200,4.316135,3.306263,5.222198,78.285097,...,75.492519,78.214006,13.420037,4849,1030,710,408,gemma_3,Gemma 3 27B,16


In [22]:
df_breakdown[['predictor', 'predictor_index', 'normalized_gain']]
#df_breakdown.columns

# First, let's see the structure
print(f"Total rows: {len(df_breakdown)}")
print(f"Rows per predictor: {df_breakdown.groupby('predictor').size()}")

# If each predictor has the same number of rows in the same order (18 reference models),
# we can pivot by adding a reference model index within each predictor group
df_breakdown['ref_model_idx'] = df_breakdown.groupby('predictor').cumcount()

# Pivot to matrix form: rows = reference models, columns = predictors
scores_matrix = df_breakdown.pivot(
    index='model', 
    columns='predictor', 
    values='normalized_gain'
)

print(scores_matrix.shape)  # Should be (18, 5)
scores_matrix


Total rows: 90
Rows per predictor: predictor
Qwen/Qwen3-32B                   18
google/gemini-3-flash-preview    18
google/gemma-3-27b-it            18
openai/gpt-5-mini                18
openai/gpt-oss-20b               18
dtype: int64
(18, 5)


predictor,Qwen/Qwen3-32B,google/gemini-3-flash-preview,google/gemma-3-27b-it,openai/gpt-5-mini,openai/gpt-oss-20b
model,,,,,
Qwen/Qwen3-0.6B_True,13.833671,7.598945,18.561585,-0.971398,12.984055
Qwen/Qwen3-1.7B_True,23.851030,6.236674,20.393927,8.428928,16.161616
Qwen/Qwen3-14B_True,37.121867,23.259153,38.014042,24.617347,40.090293
Qwen/Qwen3-32B_True,41.314357,24.411567,39.610964,22.345679,42.279570
Qwen/Qwen3-4B_True,29.464286,14.372247,25.281658,14.884471,32.530629
Qwen/Qwen3-8B_True,32.150584,21.532365,31.902439,18.985776,36.387900
anthropic/claude-haiku-4.5_medium,32.221284,12.595685,32.887454,14.987245,32.535461
anthropic/claude-opus-4.5_medium,31.541219,7.274321,37.441643,18.858733,33.660131
anthropic/claude-sonnet-4.5_medium,40.982867,17.223460,45.656918,21.114551,33.566760


In [23]:
# Rank agreement
# Kendall's W (also known as Kendall's coefficient of concordance) is a non-parametric statistic for rank correlation.
# If the test statistic W is 1, then all the survey respondents have been unanimous, and each respondent has assigned the same order to the list of concerns. If W is 0, then there is no overall trend of agreement among the respondents, and their responses may be regarded as essentially random. 
# Kendall's W makes no assumptions regarding the nature of the probability distribution (unlike PCC)
# W > 0.7 is typically considered strong agreement

In [24]:
# Assume `scores` is shape (n_reference_models, n_predictors) = (18, 5)

# 1. Kendall's W
def kendalls_w(scores):
    """scores: (n_items, n_raters)"""
    n, k = scores.shape
    # Convert to ranks
    ranks = np.apply_along_axis(stats.rankdata, 0, scores)
    # Sum of ranks for each item
    R = ranks.sum(axis=1)
    R_bar = R.mean()
    S = ((R - R_bar) ** 2).sum()
    W = 12 * S / (k**2 * (n**3 - n))
    return W

kendalls_w(scores_matrix)

np.float64(0.7470381836945305)

In [25]:
# Absolute value agreement
# Intraclass Correlation Coefficient (ICC)
# It describes how strongly units in the same group resemble each other. While it is viewed as a type of correlation, unlike most other correlation measures, it operates on data structured as groups rather than data structured as paired observations.

# Convert to numpy array
scores = scores_matrix.values  # (n_items, n_raters)

# 1. Kendall's W
def kendalls_w(scores):
    n, k = scores.shape
    ranks = np.apply_along_axis(stats.rankdata, 0, scores)
    R = ranks.sum(axis=1)
    R_bar = R.mean()
    S = ((R - R_bar) ** 2).sum()
    W = 12 * S / (k**2 * (n**3 - n))
    return W

W = kendalls_w(scores)
print(f"Kendall's W: {W:.3f}")

# 2. Average pairwise Spearman
def avg_pairwise_spearman(scores):
    n, k = scores.shape
    correlations = []
    for i in range(k):
        for j in range(i+1, k):
            rho, _ = spearmanr(scores[:, i], scores[:, j])
            correlations.append(rho)
    return np.mean(correlations), np.std(correlations), correlations

mean_rho, std_rho, all_rhos = avg_pairwise_spearman(scores)
print(f"Avg Spearman ρ: {mean_rho:.3f} ± {std_rho:.3f}")
print(f"Range: [{min(all_rhos):.3f}, {max(all_rhos):.3f}]")



df_long = df_breakdown[['ref_model_idx', 'predictor', 'normalized_gain']].copy()
df_long.columns = ['targets', 'raters', 'ratings']

icc_results = pg.intraclass_corr(
    data=df_long, 
    targets='targets', 
    raters='raters', 
    ratings='ratings'
)
print("\nICC Results:")
print(icc_results[['Type', 'ICC', 'CI95%']])


Kendall's W: 0.747
Avg Spearman ρ: 0.684 ± 0.138
Range: [0.470, 0.926]

ICC Results:
    Type       ICC         CI95%
0   ICC1  0.208862  [0.03, 0.47]
1   ICC2  0.304294   [0.06, 0.6]
2   ICC3  0.766736  [0.61, 0.89]
3  ICC1k  0.568968  [0.15, 0.82]
4  ICC2k  0.686220  [0.23, 0.88]
5  ICC3k  0.942644  [0.89, 0.98]
